# Notebook 1: FRIP Signal Generation (GEE)

This notebook computes the raw Flood-Referenced Integrated Productivity (FRIP) signal using Google Earth Engine (Python API).

**FRIP** is defined as the spatial Spearman rank correlation between:
1. JRC CEMS GLOFAS flood inundation depth (sum of 7 return periods).
2. MODIS annual net primary productivity (MOD17A3HGF, 2001–2023).

The correlation is calculated spatially within moving windows (or grid cells) corresponding to the target resolutions (5km to 100km). Only undisturbed forest pixels (JRC TMF Class 10) that are flood-connected (MERIT Hydro HND > 0) are included in the correlation.

**Outputs**:
Exports to GEE Assets:
- `FRIP_{scale}` (20 assets, cross-sectional Spearman r)
- `FRIP_Annual_{scale}` (20 assets, 23-band annual correlations)

In [ ]:
import ee
import geemap

# Initialize Earth Engine
try:
    ee.Initialize(project='quantum-bonus-434714-t2')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')

## 1. Shared Spatial Parameters

In [ ]:
# Study regions — simple bounding boxes [west, south, east, north]
CONGO_BBOX  = ee.Geometry.Rectangle([8,  -12, 35,  8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
STUDY_REGION = ee.FeatureCollection([
    ee.Feature(CONGO_BBOX,  {'basin': 'Congo'}),
    ee.Feature(AMAZON_BBOX, {'basin': 'Amazon'})
])

FOREST_MASK  = 'projects/JRC/TMF/v1_2024/TransitionMap_MainClasses'
FOREST_CLASS = 10   # continuously undisturbed since ~1982
FOREST_COVER_THRESHOLD = 0.95
SCALES       = list(range(5000, 105000, 5000))
ASSET_ROOT   = 'users/JakeWilliams844/DefaunationFromSpace'

## 2. Data Loading and Masking

In [ ]:
# 2.1 Forest Mask (Native ~30m resolution)
tmf = ee.Image(FOREST_MASK)
forest_mask = tmf.eq(FOREST_CLASS)

# 2.2 MERIT Hydro Flood Connectivity Mask (Native ~90m resolution)
merit = ee.Image("MERIT/Hydro/v1_0_1")
hnd = merit.select('hnd')
hnd_mask = hnd.gt(0)

# Combine masks at native resolution
combined_native_mask = forest_mask.updateMask(hnd_mask)

# 2.3 GLOFAS Flood Depth (Sum of 7 return periods)
glofas = ee.ImageCollection('JRC/CEMS_GLOFAS/FloodHazard/v1')
flood_depth = glofas.select('depth').sum()

# Apply native mask to flood depth
masked_flood_depth = flood_depth.updateMask(combined_native_mask)

# 2.4 MODIS Annual NPP (2001-2023)
modis = ee.ImageCollection("MODIS/061/MOD17A3HGF").select('Npp')
years = ee.List.sequence(2001, 2023)

def get_annual_npp(year):
    img = modis.filter(ee.Filter.calendarRange(year, year, 'year')).first()
    # Apply native mask to NPP
    return img.updateMask(combined_native_mask).set('year', year)

annual_npp = ee.ImageCollection.fromImages(years.map(get_annual_npp))
mean_npp = annual_npp.mean()  # For cross-sectional FRIP
npp_proj = annual_npp.first().projection()

## 3. Multiscale FRIP Computation

FRIP is a spatial Spearman correlation. We compute it by stacking the NPP and Flood Depth images, and applying `ee.Reducer.spearmanCorrelation()` inside a `reduceResolution` step for each target scale.

In [ ]:
def compute_frip_at_scale(scale):
    print(f"Setting up tasks for scale: {scale}m")
    
    # 1. Compute fractional forest cover at this scale
    # reduceResolution computes the fraction of 1s in the native mask
    forest_fraction = forest_mask.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).reproject(
        crs=npp_proj,
        scale=scale
    )
    
    # Mask out coarse pixels that have less than 95% intact forest
    scale_mask = forest_fraction.gte(FOREST_COVER_THRESHOLD)
    
    # 2. Cross-sectional FRIP (Mean NPP vs Flood Depth)
    # Rename bands because spearmanCorrelation expects unique names
    cross_stack = ee.Image.cat([
        mean_npp.rename('npp'), 
        masked_flood_depth.rename('depth')
    ])
    
    frip_cross = cross_stack.reduceResolution(
        reducer=ee.Reducer.spearmanCorrelation(),
        maxPixels=65535
    ).reproject(
        crs=npp_proj,
        scale=scale
    ).updateMask(scale_mask)
    
    # The reducer outputs two bands: 'correlation' and 'p-value'
    # We only need 'correlation'
    frip_cross_corr = frip_cross.select('correlation').rename(f'FRIP_{scale}')
    
    # 3. Annual FRIP (Annual NPP vs Flood Depth)
    def compute_annual_frip(img):
        year = img.get('year')
        annual_stack = ee.Image.cat([
            img.rename('npp'),
            masked_flood_depth.rename('depth')
        ])
        frip_ann = annual_stack.reduceResolution(
            reducer=ee.Reducer.spearmanCorrelation(),
            maxPixels=65535
        ).reproject(
            crs=npp_proj,
            scale=scale
        ).updateMask(scale_mask)
        # Rename to include year
        return frip_ann.select('correlation').rename(ee.String('FRIP_').cat(ee.Number(year).format('%04d')))
        
    frip_annual_col = annual_npp.map(compute_annual_frip)
    # Convert collection to a 23-band image
    frip_annual_img = frip_annual_col.toBands()
    # Clean up band names (removes the index prefix added by toBands)
    band_names = frip_annual_col.aggregate_array('system:index').map(lambda x: ee.String('FRIP_').cat(x))
    frip_annual_img = frip_annual_img.rename(band_names)
    
    return frip_cross_corr, frip_annual_img


## 4. Execute Asset Exports

In [ ]:
tasks = []

for scale in SCALES:
    frip_cross, frip_annual = compute_frip_at_scale(scale)
    
    # Export Cross-sectional
    task_cross = ee.batch.Export.image.toAsset(
        image=frip_cross,
        description=f'Export_FRIP_{scale}',
        assetId=f'{ASSET_ROOT}/FRIP_{scale}',
        region=STUDY_REGION.geometry(),
        scale=scale,
        crs=npp_proj,
        maxPixels=1e13
    )
    tasks.append(task_cross)
    
    # Export Annual
    task_annual = ee.batch.Export.image.toAsset(
        image=frip_annual,
        description=f'Export_FRIP_Annual_{scale}',
        assetId=f'{ASSET_ROOT}/FRIP_Annual_{scale}',
        region=STUDY_REGION.geometry(),
        scale=scale,
        crs=npp_proj,
        maxPixels=1e13
    )
    tasks.append(task_annual)

# Uncomment to start all tasks
print(f"Created {len(tasks)} export tasks.")
# for task in tasks:
#     task.start()
    
# print("All tasks started. Monitor progress in the GEE Code Editor or via ee.batch.Task.list()")